## 00 — What Are Vector Tiles?

In Module 06 we identified four pain points in our handbuilt system:
1. Slow startup (load + index all data upfront)
2. Verbose GeoJSON format
3. Whole file always resident in memory
4. No streaming — unused regions still loaded

**Vector tiles** are the data format that solves all four. This notebook explains what they are before we use the tool that generates them.

## The Core Idea — Pre-Sliced, Pre-Indexed Data

Instead of four large files that cover the whole world, a vector tile pyramid pre-slices the world into thousands of small tiles — one per `{zoom}/{x}/{y}` address — before the user ever opens the map.

```
Zoom 0: 1 tile  (whole world)
Zoom 1: 4 tiles (quadrants)
Zoom 2: 16 tiles
...
Zoom 14: 268 million tiles (most empty)
```

When the user views a map, only the tiles that are currently visible are fetched. A user in Paris at zoom 12 receives ~12 tiles covering roughly 5km × 5km each. Siberia is never touched.

## How Each Tile Maps to Our System

Every choice we made manually now happens automatically inside the tile generator:

| What we built | What the tile system does |
|---------------|---------------------------|
| 4 LOD files at fixed epsilons | Per-zoom simplification baked into each tile |
| Grid index bucketing features into cells | Each tile IS a cell — features are pre-bucketed by definition |
| Viewport bbox culling | Each tile covers a fixed bbox — fetching only nearby tiles IS the cull |
| Zoom decision function | The tile URL scheme `/{z}/{x}/{y}` carries the zoom level |
| GeoJSON text format | MVT binary encoding — coordinates as integers, ~5× smaller |
| Whole file loaded at startup | Each tile fetched on demand, ~50–200 KB each |

## The Tile Coordinate System

Tiles use `(z, x, y)` addressing. At zoom `z`, the world is divided into a `2^z × 2^z` grid.

Given a longitude/latitude, we can compute its tile address:

In [2]:
import math

def lon_lat_to_tile(lon, lat, zoom):
    """Return the (z, x, y) tile address for a geographic point at a given zoom."""
    n = 2 ** zoom
    x = int((lon + 180) / 360 * n)
    lat_rad = math.radians(lat)
    y = int((1 - math.log(math.tan(lat_rad) + 1 / math.cos(lat_rad)) / math.pi) / 2 * n)
    return zoom, x, y

# Paris
for zoom in [2, 5, 8, 12]:
    z, x, y = lon_lat_to_tile(2.35, 48.86, zoom)
    print(f"  zoom {z:>2}  tile ({z}/{x}/{y})")

  zoom  2  tile (2/2/1)
  zoom  5  tile (5/16/11)
  zoom  8  tile (8/129/88)
  zoom 12  tile (12/2074/1409)


## The MVT Binary Format

Mapbox Vector Tiles (MVT) store geometry as integers instead of floating-point text.

Each tile has a local coordinate space of 4096 × 4096 units. A coordinate like `(48.8566, 2.3522)` is projected into this space and stored as two small integers (e.g., `(2047, 1803)`).

This gives:
- **5–10× smaller files** vs. GeoJSON (integers compress better than decimal strings)
- **Faster parse** (no string-to-float conversion)
- **Lossy but controlled precision** (4096 units per tile at zoom 14 ≈ 2m resolution)

## The PMTiles Format

Traditionally, tile pyramids were stored in SQLite databases (`.mbtiles`) or as millions of individual files on a server.

**PMTiles** is a newer single-file format that stores the entire tile pyramid in one `.pmtiles` file, arranged so that spatially nearby tiles are stored close together on disk. A client can fetch just the tiles it needs using HTTP range requests — no tile server required, just a static file on any CDN.

For our purposes: `tippecanoe` can output either `.mbtiles` or `.pmtiles`.

## Exercise A

At zoom 12, the world is divided into `2^12 × 2^12 = 4096 × 4096 = ~16.7 million` tiles.

1. How many tiles cover Western Europe at zoom 12? (Approximate using the bounding box [-10, 35, 30, 60])
2. If each tile is 100 KB on average, how much data would the user need to download to view all of Western Europe at zoom 12?

Compare that to loading our `extra_fine` GeoJSON for the same region.

In [3]:
# Calculate tile count for Western Europe at zoom 12
# Estimate download size vs. GeoJSON approach
import json
from pathlib import Path

bbox = [-10, 35, 30, 60]  # west, south, east, north
west, south, east, north = bbox
zoom = 12

# lon_lat_to_tile returns y values that increase southward, so use north for
# the top tile and south for the bottom tile.
_, x_min, y_bottom = lon_lat_to_tile(west, south, zoom)
_, x_max, y_top = lon_lat_to_tile(east, north, zoom)

tile_columns = x_max - x_min + 1
tile_rows = y_bottom - y_top + 1
tile_count = tile_columns * tile_rows

avg_tile_kb = 100
tile_download_mb = tile_count * avg_tile_kb / 1_000
tile_download_gb = tile_download_mb / 1_000

extra_fine_path = Path("../../data/lod/railroads_extra_fine.geojson")
if not extra_fine_path.exists():
    extra_fine_path = Path("Assignments_Completed/03-Data_Manager/data/lod/railroads_extra_fine.geojson")

with open(extra_fine_path) as f:
    extra_fine = json.load(f)

def feature_bbox(feature):
    coords = feature["geometry"]["coordinates"]
    xs = [coord[0] for coord in coords]
    ys = [coord[1] for coord in coords]
    return min(xs), min(ys), max(xs), max(ys)

def intersects_bbox(feature, query_bbox):
    q_west, q_south, q_east, q_north = query_bbox
    f_west, f_south, f_east, f_north = feature_bbox(feature)
    return not (f_east < q_west or f_west > q_east or f_north < q_south or f_south > q_north)

western_europe_features = [
    feature for feature in extra_fine["features"]
    if intersects_bbox(feature, bbox)
]
western_europe_geojson = {
    "type": "FeatureCollection",
    "features": western_europe_features,
}
western_europe_geojson_mb = len(json.dumps(western_europe_geojson).encode("utf-8")) / 1_000_000
full_extra_fine_mb = extra_fine_path.stat().st_size / 1_000_000

print(f"Tile range: x {x_min}-{x_max}, y {y_top}-{y_bottom}")
print(f"Tile grid covering bbox: {tile_columns:,} columns x {tile_rows:,} rows")
print(f"Tiles covering Western Europe at zoom {zoom}: {tile_count:,}")
print(f"Estimated tile download at 100 KB/tile: {tile_download_mb:,.1f} MB ({tile_download_gb:.2f} GB)")
print(f"Extra-fine GeoJSON features intersecting bbox: {len(western_europe_features):,}")
print(f"Estimated same-region extra-fine GeoJSON size: {western_europe_geojson_mb:.2f} MB")
print(f"Full extra-fine GeoJSON file size: {full_extra_fine_mb:.2f} MB")

# Conclusion: all of Western Europe at zoom 12 is much too large to fetch as tiles
# all at once: about 197,904 tiles, or about 19.79 GB at 100 KB per tile.
# Vector tiles are useful because the client normally fetches only the handful of
# visible tiles, not every zoom-12 tile in a continent-sized bounding box.

Tile range: x 1934-2389, y 1189-1622
Tile grid covering bbox: 456 columns x 434 rows
Tiles covering Western Europe at zoom 12: 197,904
Estimated tile download at 100 KB/tile: 19,790.4 MB (19.79 GB)
Extra-fine GeoJSON features intersecting bbox: 9,836
Estimated same-region extra-fine GeoJSON size: 6.45 MB
Full extra-fine GeoJSON file size: 18.98 MB


## Exercise B

The tile coordinate formula uses the Web Mercator projection — the same projection used by Google Maps, OpenStreetMap, and virtually all web maps.

Web Mercator distorts areas significantly near the poles. Greenland appears roughly the same size as Africa on a Web Mercator map, even though Africa is ~14× larger.

Does this distortion affect the **accuracy** of our railroad visualization? Explain why or why not in 3–4 sentences.

In [4]:
# Write your answer as a markdown cell or comment
# Web Mercator distortion does not change the underlying railroad coordinates;
# it changes how those coordinates are projected onto the screen. For a visual
# map overlay, the railroad lines remain correctly aligned with the basemap
# because both use the same projection. The distortion would matter if we tried
# to measure real-world area, distance, or density directly from the screen.
# For showing where railroads are located, it is accurate enough for normal web
# map use, especially away from the extreme polar regions.

## Check Your Understanding

The tile grid at zoom 14 has ~268 million possible tile addresses. Most tiles — over oceans, deserts, and polar regions — contain no data.

Both `.mbtiles` (SQLite) and `.pmtiles` (single file) only store non-empty tiles. Why is this critical, and how does it relate to the `scalerank` filtering decision we made in our LOD pipeline?

This is critical because the possible tile address space is enormous, but the useful data is sparse. Storing empty tiles would waste disk space, transfer bandwidth, and lookup time on places where there is nothing to draw. Only storing non-empty tiles is the tile-system version of the same idea behind our `scalerank` filter: keep the data that can affect the current map and drop data that only adds cost. The difference is that empty-tile skipping removes spatially irrelevant tiles, while `scalerank` removes low-importance railroad features at coarse zoom levels.

---

## Next

In [01 — Using Tippecanoe](./01-Using_Tippecanoe.ipynb), we run `tippecanoe` on the raw railroad GeoJSON and map each of its flags to decisions we already made.